# Main agents in siRNA therapeutics — metadata fetch, scope filter + analysis

This notebook runs a three-step workflow:

1. **`epo_api_full_meta.py`** reads a CSV of patent IDs and fetches full bibliographic metadata from the EPO OPS `/biblio` endpoint, writing a `*_metadata.csv`. Each row now also carries `Title`, `Abstract`, `IPC` and `CPC` (all taken from the same `/biblio` call, so no extra API requests).
2. **`epo_analise.py` → `filter_siRNA_patents`** restricts that metadata to therapeutic-siRNA families: a classification gate (CPC/IPC in the in-scope set, excluding aptamer/immunomodulatory codes) and a text gate (drops competing-technology, purely-diagnostic, and agricultural/veterinary patents by Title/Abstract). It writes a `*_filtered.csv`.
3. **`epo_analise.py` → `generate_timeline` / `calculate_top_applicants`** run on the filtered file to produce the innovation timeline and the ranking of top applicants.

All `.py` files must sit in the same directory as this notebook.

## 1. Credentials

Register an application at the EPO Open Patent Services portal to obtain a consumer key and secret.

In [1]:
from getpass import getpass

# Securely prompt the user for credentials at runtime
EPO_CONSUMER_KEY = getpass("Enter your EPO Consumer Key: ")
EPO_CONSUMER_SECRET = getpass("Enter your EPO Consumer Secret: ")

# The CSV filename doesn't need to be hidden, so it stays a standard string
INPUT_IDS_CSV = "EPO_siRNA_IDs_2001_2026_terms_only.csv"

## 2. Fetch the metadata

This makes live API calls and can take a while for thousands of IDs (the EPO rate limit is roughly 10 requests/minute). The output CSV name is derived automatically by appending `_metadata`, or you can set `output_csv` explicitly. The response now includes the title, abstract, IPC and CPC classifications used by the filter in step 3.

In [2]:
from epo_api_full_meta import fetch_biblio_from_csv

df_meta = fetch_biblio_from_csv(
    INPUT_IDS_CSV,
    consumer_key=EPO_CONSUMER_KEY,
    consumer_secret=EPO_CONSUMER_SECRET,
    # output_csv="siRNA_metadata.csv",  # optional: choose the output name
)

# Default output path when output_csv is not given:
METADATA_CSV = INPUT_IDS_CSV.replace(".csv", "_metadata.csv")
df_meta.head()


=== STARTING EPO METADATA FETCH ===
[INFO] Input file : EPO_siRNA_IDs_2001_2026_terms_only.csv
[INFO] Patent IDs : 29628
[INFO] Batches    : 297 x up to 100 IDs per batch

[AUTH] Generating a new EPO access token...
  [QUOTA] hour    1.5 MB / ~450 MB ( 0.3%) | week   285.5 MB | light green
[INFO] Batch 1/297 done — 100 record(s) so far.
  [QUOTA] hour    2.6 MB / ~450 MB ( 0.6%) | week   286.6 MB | light green
[INFO] Batch 2/297 done — 200 record(s) so far.
  [QUOTA] hour    3.6 MB / ~450 MB ( 0.8%) | week   287.6 MB | light green
[INFO] Batch 3/297 done — 300 record(s) so far.
  [QUOTA] hour    4.9 MB / ~450 MB ( 1.1%) | week   288.9 MB | light green
[INFO] Batch 4/297 done — 400 record(s) so far.
  [QUOTA] hour    6.6 MB / ~450 MB ( 1.5%) | week   290.6 MB | light green
[INFO] Batch 5/297 done — 500 record(s) so far.
  [QUOTA] hour    7.9 MB / ~450 MB ( 1.8%) | week   291.9 MB | light green
[INFO] Batch 6/297 done — 600 record(s) so far.
  [QUOTA] hour    9.4 MB / ~450 MB ( 2.1%) | 

,Patent_ID,Country,Number,Kind,Family_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,IPC,CPC
8562,US7285537B1,US,7285537,B1,23218662,19811023,20071023,"ISIS PHARMACEUTICALS, INC.",Oligonucleotide therapeutic agent and methods ...,For use in controlling biologic functions in a...,A01H1/00 A61K31/70 A61K31/7088 A61K31/7135 A61...,A61K31/7088 A61K31/7135 C07H21/00 C12N15/1131 ...
2972,US8097405B1,US,8097405,B1,23546604,19820623,20120117,STAVRIANOPOULOS JANNIS G | ENZO BIOCHEM INC | ...,Nucleic acid sequencing processes using non-ra...,A process for determining the sequence of nucl...,A61K47/48 C07B59/00 C07H19/04 C07H19/10 C07H19...,A61K47/62 C07B59/008 C07H19/04 C07H19/10 C07K1...
3506,USRE43096E,US,RE43096,E,27557712,19840116,20120110,CALIFORNIA INST OF TECHN | SMITH LLOYD M | HUN...,Tagged extendable primers and extension products,[0000] This invention provides a duplex compri...,C12Q1/68 G01N27/447,C12Q1/6816 C12Q1/6869 G01N27/44721 G01N27/44726
8104,US6200748B1,US,6200748,B1,46251268,19840116,20010313,CALIFORNIA INSTITUTE OF TECHNOLOGY,Tagged extendable primers and extension products,This invention provides a duplex comprising an...,C12Q1/68 G01N27/447,C12Q1/6816 C12Q1/6869 G01N27/44721 G01N27/44726
7662,US6180104B1,US,6180104,B1,27079355,19840301,20010130,THE BOARD OF TRUSTEES OF THE LELAND STANFORD J...,T cell receptor beta subunit,Oligonucleotide sequences are provided coding ...,A61K35/12 A61K35/14 A61K35/74 A61K38/00 A61K38...,A61P37/00 A61P37/02 A61P43/00 C07K14/7051 C07K...


## 3. Filter to therapeutic-siRNA scope

`filter_siRNA_patents` keeps only families whose CPC/IPC codes are in the in-scope siRNA set (mirroring the OPS extraction query, and excluding the aptamer/immunomodulatory codes `C12N15/115` and `C12N15/117`), then drops families whose Title/Abstract reveal a competing technology (CRISPR, ASO, miRNA, …), a purely diagnostic scope, or an agricultural/veterinary scope. It prints a per-gate breakdown and writes a `*_filtered.csv`.

In [3]:
from epo_analise import filter_siRNA_patents

df_filtered = filter_siRNA_patents(METADATA_CSV)

# Default output path written by the filter:
import os, re
FILTERED_CSV = re.sub(r"\.csv$", "_filtered.csv", os.path.basename(METADATA_CSV))
df_filtered.head()


[INFO] Filtering 'EPO_siRNA_IDs_2001_2026_terms_only_metadata.csv' to therapeutic-siRNA scope...
[INFO] Input families            : 29627
[INFO] Out of classification scope: 14377
[INFO] Dropped — competing tech   : 4336
[INFO] Dropped — pure diagnostic  : 353
[INFO] Dropped — agri/veterinary  : 1124
[INFO] Retained                   : 9597
[SUCCESS] Filtered metadata saved: 'EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.csv'


,Patent_ID,Country,Number,Kind,Family_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,IPC,CPC
0,US7285537B1,US,7285537,B1,23218662,19811023,20071023,"ISIS PHARMACEUTICALS, INC.",Oligonucleotide therapeutic agent and methods ...,For use in controlling biologic functions in a...,A01H1/00 A61K31/70 A61K31/7088 A61K31/7135 A61...,A61K31/7088 A61K31/7135 C07H21/00 C12N15/1131 ...
1,US8097405B1,US,8097405,B1,23546604,19820623,20120117,STAVRIANOPOULOS JANNIS G | ENZO BIOCHEM INC | ...,Nucleic acid sequencing processes using non-ra...,A process for determining the sequence of nucl...,A61K47/48 C07B59/00 C07H19/04 C07H19/10 C07H19...,A61K47/62 C07B59/008 C07H19/04 C07H19/10 C07K1...
11,US2003191078A1,US,2003191078,A1,28679193,19860523,20031009,"HYBRIDON, INC.",Inhibition of infectious agents by exogenous o...,Inhibition of replication of an infectious age...,C07H21/00 C12N15/113 C12Q1/70 G01N33/569,A61K31/70 C07H21/00 C12N15/1131 C12N15/1132 C1...
22,US2003186911A1,US,2003186911,A1,28457989,19870710,20031002,"HYBRIDON, INC.",Inhibition of infectious agents by exogenous o...,Inhibition of replication of an infectious age...,C07H21/00 C12N15/113 C12Q1/70,A61K31/70 C07H21/00 C12N15/1131 C12N15/1132 C1...
24,US6197944B1,US,6197944,B1,22425523,19871130,20010306,"INTEGRATED DNA TECHNOLOGIES, INC.",DNA molecules stabilized by modifications of t...,The use of oligodeoxynucleotides modified at t...,A61K48/00 C07H19/00 C07H21/00 C12N15/09 C12N15...,C07H21/00 C12N15/113 C12Q1/6813 C12Q1/6848 C12...


## 4. Analyse the metadata

`generate_timeline` charts unique patent families by priority year. `calculate_top_applicants` cleans and consolidates applicant names, then ranks them by number of unique families. Each function writes a CSV and a PDF chart into the working directory. Both run on the **filtered** file.

In [6]:
from epo_analise import generate_timeline, calculate_top_applicants

generate_timeline(FILTERED_CSV, start_year=2001, end_year=2024)

df_top = calculate_top_applicants(FILTERED_CSV, top_n=5)
df_top.head(30)


[INFO] Generating timeline for EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.csv (2001-2024) using ONLY Priority Date...
[SUCCESS] Timeline generated. CSV: 'timeline_years_EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.csv' | PDF: 'chart_timeline_EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.pdf'

[INFO] Calculating Top 5 Applicants for EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.csv...
[SUCCESS] Top 5 generated. CSV: 'top5_EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.csv' | PDF: 'chart_top_applicants_EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.pdf'



,Company,Number_of_Patents,Earliest_Innovation
155,ALNYLAM PHARMACEUTICALS,346,19990130
2326,ISIS PHARMACEUTICALS,195,19811023
252,ARROWHEAD PHARMACEUTICALS,103,20020904
5330,UNIVERSITY OF MASSACHUSETTS,102,19920226
1125,DICERNA PHARMACEUTICALS,66,20080421


In [8]:
from epo_analise import generate_timeline, calculate_top_applicants

generate_timeline(FILTERED_CSV, start_year=2002, end_year=2024)

df_top = calculate_top_applicants(FILTERED_CSV, top_n=5)
df_top.head(30)


[INFO] Generating timeline for EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.csv (2002-2024) using ONLY Priority Date...
[SUCCESS] Timeline generated. CSV: 'timeline_years_EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.csv' | PDF: 'chart_timeline_EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.pdf'

[INFO] Calculating Top 5 Applicants for EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.csv...
[SUCCESS] Top 5 generated. CSV: 'top5_EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.csv' | PDF: 'chart_top_applicants_EPO_siRNA_IDs_2001_2026_terms_only_metadata_filtered.pdf'



,Company,Number_of_Patents,Earliest_Innovation
155,ALNYLAM PHARMACEUTICALS,346,19990130
2326,ISIS PHARMACEUTICALS,195,19811023
252,ARROWHEAD PHARMACEUTICALS,103,20020904
5330,UNIVERSITY OF MASSACHUSETTS,102,19920226
1125,DICERNA PHARMACEUTICALS,66,20080421
